In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from modeling.dataloader import INSPIRE
from modeling.model import *
from modeling.utils import Logger

from tqdm import tqdm
import numpy as np
import json
import random
import pandas as pd
import os
import datetime

from sklearn.metrics import accuracy_score, recall_score, \
                            precision_score, f1_score, roc_auc_score, \
                            average_precision_score

In [3]:
# 初始化 Logger
task_name = 'postop_stroke'
logger = Logger(save_dir='checkpoints', model_name=task_name, resume=True)

未找到之前的训练记录，将开始新的训练


In [4]:
y_index = 6
device = torch.device("cuda:0")
input_size_list = [37 + 36, 29 + 16, 53-4]
hidden_size_list = [256, 256, 256]
output_size_list = [1]
type_list = ["cat"]
model_list = ["lstm", "lstm", "mlp"]

In [5]:
ds = pd.read_csv("/home/luojiawei/inspire_benchmark_data/operation_.csv", header=0)
train_datasets = []
for i in [0, 1]:
    train_datasets.append(INSPIRE(
        "/home/luojiawei/inspire_benchmark_data/all_op_id/",
        ds[(ds['dataset'] == 1) & (ds[task_name] == i)],
        id_col="op_id",
        param_path="/home/luojiawei/inspire_benchmark/param_folder"
    ))

valid_datasets = []
for i in [0, 1]:
    valid_datasets.append(INSPIRE(
        "/home/luojiawei/inspire_benchmark_data/all_op_id/",
        ds[(ds['dataset'] == 3) & (ds[task_name] == i)],
        id_col="op_id",
        param_path="/home/luojiawei/inspire_benchmark/param_folder"
    ))


Number of samples: 70301
Number of samples: 700
Number of samples: 10003
Number of samples: 104


In [6]:
model = PredModel(input_size_list, hidden_size_list, model_list, output_size_list, type_list)
model = model.to(device)

In [7]:
# -------- 训练参数 ------------
EPOCHS = 100
lr = 0.001
weight_decay = 0.001
batch_size = 300
batch_size_val = 300
best_auc = 0.5
no_improvement_count = 0
max_iter_train = 10
max_iter_val = 10
tol_count = 2
optimizer = optim.Adam(model.parameters(), 
                       lr=lr, 
                       weight_decay=weight_decay)
if output_size_list[0] > 1:
    loss_fn = nn.CrossEntropyLoss().to(device)
else:
    loss_fn = nn.BCEWithLogitsLoss()

In [8]:
print("开始训练...")
for epoch in range(EPOCHS):
    model.train()
    epoch_train_loss = 0.0
    
    # 创建训练数据加载器
    data_loaders = [dataset.iterate_batch(batch_size, normalize=True) for dataset in train_datasets]
    
    # 使用tqdm显示counter的进度
    pbar = tqdm(total=max_iter_train, desc=f"Epoch {epoch+1}/{EPOCHS}")
    counter = 1
    while counter <= max_iter_train:
        running_loss = 0.0
        y_true, y_pred = [], []
        
        # 获取每个数据加载器的下一批数据
        data_batches = []
        for loader_index, loader in enumerate(data_loaders):
            try:
                batch, ids = next(loader)
            except StopIteration:
                # 重新创建生成器
                loader = train_datasets[loader_index].iterate_batch(batch_size, normalize=True)
                batch, ids = next(loader)
                # 更新 data_loaders 列表中的生成器
                data_loaders[loader_index] = loader
            data_batches.append(batch)
        
        # 处理所有批次的数据
        for batch in data_batches:
            for i in range(len(batch)):  # 移除这里的tqdm
                datas = batch[i]
                
                lab, mask_lab, vit, mask_vit, ward_vit, mask_ward_vit, \
                                _, _, x_s, \
                                _, _, y_static, y_mask1 = datas
     
               
                # 将 lab 和 mask_lab 合并
                lab_c = torch.cat((lab, mask_lab), dim=-1).unsqueeze(0).to(device)
                
                # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                # vit_c = torch.cat((vit, mask_vit), dim=-1).unsqueeze(0).to(device)

                # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                ward_vit_c = torch.cat((ward_vit, mask_ward_vit), dim=-1).unsqueeze(0).to(device)
                
                # 将 x_s 移动到设备
                x_s = x_s[:,:-4].to(device)
                
                # 更新模型输入
                inputs = [lab_c, ward_vit_c, x_s]
                
                # 获取模型预测
                yhat_list = model(inputs)
                y_pred.append(yhat_list[0])
                y_true.append(y_static[:, y_index:(y_index+1)])

        y_pred = torch.cat(y_pred, dim=0)
        y_true = torch.cat(y_true, dim=0)
        if output_size_list[0] == 1:
            y_true = (y_true > 0).float().to(device)
        else:
            y_true = y_true.to(torch.int64).to(device).reshape(-1)
        loss = loss_fn(y_pred, y_true)
        running_loss += loss.cpu().item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += running_loss
        
        # 更新进度条，并显示当前loss
        pbar.set_postfix({"Loss": f"{running_loss:.6f}"})
        pbar.update(1)
        
        counter += 1
    
    pbar.close()
    print(f"Epoch {epoch+1}/{EPOCHS} 完成, 平均训练损失: {epoch_train_loss/counter:.6f}")

    # 验证阶段
    print("开始在验证集上测试...")
    model.eval()
    running_loss = 0.0
    y_true, y_pred = [], []
    
    with torch.no_grad():
        # 创建验证数据加载器
        data_loaders = [dataset.iterate_batch(batch_size_val, normalize=True) for dataset in valid_datasets]
        
        # 使用tqdm显示验证进度
        pbar_val = tqdm(total=max_iter_val, desc="验证")
        counter = 1
        while counter <= max_iter_val:
            for loader in data_loaders:
                try:
                    data_batch, ids = next(loader)
                except StopIteration:
                    break

                for i in range(len(data_batch)):  # 移除这里的tqdm
                    datas = data_batch[i]
                    
                    lab, mask_lab, vit, mask_vit, ward_vit, mask_ward_vit, \
                                    _, _, x_s, \
                                    _, _, y_static, y_mask1 = datas
        
                
                    # 将 lab 和 mask_lab 合并
                    lab_c = torch.cat((lab, mask_lab), dim=-1).unsqueeze(0).to(device)
                    
                    # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                    # vit_c = torch.cat((vit, mask_vit), dim=-1).unsqueeze(0).to(device)

                    # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                    ward_vit_c = torch.cat((ward_vit, mask_ward_vit), dim=-1).unsqueeze(0).to(device)
                    
                    # 将 x_s 移动到设备
                    x_s = x_s[:,:-4].to(device)
                    
                    # 更新模型输入
                    inputs = [lab_c, ward_vit_c, x_s]
                    
                    # 获取模型预测
                    yhat_list = model(inputs)
                    y_pred.append(yhat_list[0])
                    y_true.append(y_static[:, y_index:(y_index+1)])

            pbar_val.update(1)
            counter += 1
        
        pbar_val.close()

        y_pred = torch.cat(y_pred, dim=0)
        y_true = torch.cat(y_true, dim=0)
        if output_size_list[0] == 1:
            y_true = (y_true > 0).float().to(device)
        else:
            y_true = y_true.to(torch.int64).to(device).reshape(-1)
        loss = loss_fn(y_pred, y_true)
        running_loss += loss.cpu().item()
        
        # 将模型输出转换为概率
        y_pred_prob = y_pred.cpu().numpy()
        y_true_np = y_true.cpu().numpy()

        # 计算AUC
        if output_size_list[0] == 1:
            auc = roc_auc_score(y_true_np, y_pred_prob)
        else:
            auc = roc_auc_score(y_true_np, y_pred_prob, average='macro', multi_class='ovr')
        print("Valid loss: {:.6f}, AUC: {:.6f}".format(running_loss, auc))

        # 使用记录器保存检查点
        is_best = logger.update_best_metrics(epoch, running_loss, auc)
        logger.save_checkpoint(
            model=model,
            epoch=epoch,
            train_loss=running_loss,
            valid_loss=running_loss,
            auc=auc,
            is_best=is_best,
            optimizer=optimizer
        )
        
        if is_best:
            no_improvement_count = 0
            print("模型已更新")
        else:
            no_improvement_count += 1
        
        if no_improvement_count == tol_count:
            print(f"Early stopping at epoch {epoch}")
            break


开始训练...


Epoch 1/100: 100%|██████████| 10/10 [02:37<00:00, 15.79s/it, Loss=0.649582]


Epoch 1/100 完成, 平均训练损失: 0.626527
开始在验证集上测试...


验证: 100%|██████████| 10/10 [01:25<00:00,  8.54s/it]


Valid loss: 0.718692, AUC: 0.861696
模型已更新


...

Valid loss: 0.713290, AUC: 0.932026


Epoch 5/100: 100%|██████████| 10/10 [02:38<00:00, 15.83s/it, Loss=0.560409]


Epoch 5/100 完成, 平均训练损失: 0.534207
开始在验证集上测试...


验证: 100%|██████████| 10/10 [01:24<00:00,  8.49s/it]

Valid loss: 0.712089, AUC: 0.934240
Early stopping at epoch 4


In [9]:
# 在训练循环结束后（early stopping 或完成所有 epoch）添加测试代码
print("开始测试...")

# 创建测试数据集 - 修改为与训练集相同的格式
dataset_te = INSPIRE(
    "/home/luojiawei/inspire_benchmark_data/all_op_id/",
    ds[ds['dataset'] == 2],  # 测试集
    id_col="op_id",
    param_path="/home/luojiawei/inspire_benchmark/param_folder"
)

# 使用logger加载最优模型
print("加载最优模型...")
model = logger.load_best_model(model)

running_loss = 0.0
y_true, y_pred = [], []
sample_ids = []  # 存储样本ID

model.eval()  # 设置为评估模式
with torch.no_grad():
    # 使用dataset_te的样本数来展示进度
    for i in tqdm(range(dataset_te.len()), total=dataset_te.len(), desc="测试进度"):
        datas = dataset_te.get_1data(i, normalize=True)
        
        lab, mask_lab, vit, mask_vit, ward_vit, mask_ward_vit, \
                        t_vit, t_list, x_s, \
                        y_mat, y_mask, y_static, y_mask1 = datas
        
        # 获取样本ID
        sample_id = dataset_te.all_id[i]
        sample_ids.append(sample_id)
        
        # 将 lab 和 mask_lab 合并
        lab_c = torch.cat((lab, mask_lab), dim=-1).unsqueeze(0).to(device)
        
        # 将筛选后的 ward_vit 和 mask_ward_vit 合并，并添加维度
        ward_vit_c = torch.cat((ward_vit, mask_ward_vit), dim=-1).unsqueeze(0).to(device)
        
        # 将 x_s 移动到设备
        x_s = x_s[:,:-4].to(device)
        
        # 更新模型输入
        inputs = [lab_c, ward_vit_c, x_s]
        
        # 获取模型预测
        yhat_list = model(inputs)
        y_pred.append(yhat_list[0])
        y_true.append(y_static[:, y_index:(y_index+1)])

    y_pred = torch.cat(y_pred, dim=0)
    y_true = torch.cat(y_true, dim=0)
    if output_size_list[0] == 1:
        y_true = (y_true > 0).float().to(device)
    else:
        y_true = y_true.to(torch.int64).reshape(-1).to(device)
    loss = loss_fn(y_pred, y_true)
    running_loss += loss.cpu().item()

# 将模型输出转换为概率
y_pred_prob = y_pred.cpu().numpy()  # 模型输出已经是概率了
y_true_np = y_true.cpu().numpy()

print(f"测试损失: {running_loss}")
print(logger.get_training_summary())  # 打印训练摘要信息

# 创建DataFrame保存预测结果
results_df = pd.DataFrame({
    'op_id': sample_ids,
    'y_true': y_true_np.flatten(),
    'y_pred_prob_0': y_pred_prob.flatten()
})


开始测试...
Number of samples: 20259
加载最优模型...
加载最佳模型 (epoch 2, AUC: 0.9348)


测试进度: 100%|██████████| 20259/20259 [09:38<00:00, 35.01it/s]

测试损失: 0.7289078831672668
最佳模型: Epoch 2, AUC: 0.9348, Loss: 0.721221


In [10]:
# 保存结果到CSV文件 - 修复datetime.now()的使用
from datetime import datetime  # 正确导入datetime类
results_path = f"results/{task_name}_test_preop_minn.csv"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
results_df.to_csv(results_path, index=False)
print(f"预测结果已保存到: {results_path}")

预测结果已保存到: results/postop_stroke_test_preop_minn.csv


In [11]:
import pandas as pd
df = pd.read_csv("/home/luojiawei/inspire_benchmark_data/operation_imputed.csv",header=0)
d_static = pd.read_excel("/home/luojiawei/inspire_benchmark/data_preprocessing/var_dict.xlsx", sheet_name=2)
d_lab = pd.read_excel("/home/luojiawei/inspire_benchmark/data_preprocessing/var_dict.xlsx", sheet_name=0)
d_ward_vit = pd.read_excel("/home/luojiawei/inspire_benchmark/data_preprocessing/var_dict.xlsx", sheet_name=3)


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. 首先将y列分离出来，并找出非NA的索引
train_mask = (~pd.isna(df[df['dataset'] == 1][task_name]))
valid_mask = (~pd.isna(df[df['dataset'] == 3][task_name]))
test_mask = (~pd.isna(df[df['dataset'] == 2][task_name]))

# 打印一下每个数据集中NA的数量
print(f"训练集中NA数量: {(~train_mask).sum()}")
print(f"验证集中NA数量: {(~valid_mask).sum()}")
print(f"测试集中NA数量: {(~test_mask).sum()}")

# 获取y值（去除NA）
y_train = df[df['dataset'] == 1][task_name][train_mask].values
y_valid = df[df['dataset'] == 3][task_name][valid_mask].values
y_test = df[df['dataset'] == 2][task_name][test_mask].values

d_static = d_static[~d_static['itemid'].str.contains('_duration')]

# 2. 准备特征列
# 从d_static中获取特征类型信息
numeric_features = []
categorical_features = []

# 添加ward_vit_和lab_特征
ward_cols = [col for col in df.columns if col.startswith('ward_vit_')]
lab_cols = [col for col in df.columns if col.startswith('lab_')]

# 处理d_static中的特征
for _, row in d_static.iterrows():
    item_id = row['itemid']
    value_type = row['value_type']
    
    # 根据类型分类特征
    if value_type in ['num', 'bin']:  # bin和num都归为数值型
        numeric_features.append(item_id)
    elif value_type in ['cat', 'ord']:  # ord和cat都归为离散型
        categorical_features.append(item_id)

# 处理lab_cols和ward_cols
for col in lab_cols + ward_cols:
    # 从右边分割一次，提取基本特征名（去掉最后的统计量后缀）
    parts = col.split('_')
    stat_suffix = parts[-1]
    
    # 检查是否包含统计量后缀
    if stat_suffix in ['mean', 'median', 'max', 'min', 'sd', 'sum', 'iqr', 'mode', 'any']:
        # 提取不带前缀和后缀的基本特征名
        if col.startswith('lab_'):
            # 对于lab特征，去掉'lab_'前缀和统计量后缀
            base_feature = '_'.join(parts[1:-1])
        elif col.startswith('ward_vit_'):
            # 对于ward_vit特征，去掉'ward_vit_'前缀和统计量后缀
            base_feature = '_'.join(parts[2:-1])
        
        # 查找该特征在d_lab或d_ward_vit中的类型
        feature_type = None
        
        # 在d_lab中查找
        if col.startswith('lab_'):
            for _, row in d_lab.iterrows():
                if row['itemid'] == base_feature:
                    feature_type = row['value_type']
                    break
                
        # 在d_ward_vit中查找
        elif col.startswith('ward_vit_'):
            for _, row in d_ward_vit.iterrows():
                if row['itemid'] == base_feature:
                    feature_type = row['value_type']
                    break
        
        # 根据类型分类特征
        if feature_type:
            if feature_type in ['num', 'bin']:  # bin和num都归为数值型
                numeric_features.append(col)
            elif feature_type in ['cat', 'ord']:  # ord和cat都归为离散型
                categorical_features.append(col)
        else:
            # 如果在字典中找不到类型信息，默认为数值型
            numeric_features.append(col)
    else:
        # 如果没有统计量后缀，默认为数值型
        numeric_features.append(col)

# 确保没有重复
numeric_features = list(set(numeric_features))
categorical_features = list(set(categorical_features))

# 3. 创建预处理器
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse=False, handle_unknown='ignore'), categorical_features)
    ])

# 4. 对训练集进行拟合和转换（使用非NA的索引）
X_train = preprocessor.fit_transform(df[df['dataset'] == 1][train_mask])

# 5. 使用相同的参数转换验证集和测试集（使用非NA的索引）
X_valid = preprocessor.transform(df[df['dataset'] == 3][valid_mask])
X_test = preprocessor.transform(df[df['dataset'] == 2][test_mask])

# 6. 获取特征名称
numeric_feature_names = numeric_features
categorical_feature_names = []
for i, feature in enumerate(categorical_features):
    categories = preprocessor.named_transformers_['cat'].categories_[i][1:]  # drop='first'所以从第二个开始
    categorical_feature_names.extend([f"{feature}_{cat}" for cat in categories])

feature_names = numeric_feature_names + categorical_feature_names

训练集中NA数量: 0
验证集中NA数量: 0
测试集中NA数量: 0


In [13]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.neural_network import MLPClassifier

# 数据预处理函数
def preprocess_data(X):
    # 检查并打印异常值信息
    print("数据检查:")
    print(f"NaN 值数量: {np.isnan(X).sum()}")
    print(f"无穷大值数量: {np.isinf(X).sum()}")

    # 将无穷大值替换为该列的最大有限值
    X = np.nan_to_num(X, nan=0, posinf=np.finfo(np.float32).max, neginf=np.finfo(np.float32).min)
    
    # 确保数据在float32范围内
    X = X.astype(np.float32)
        
    return X

# 预处理训练集、验证集和测试集
X_train_processed = preprocess_data(X_train)
X_valid_processed = preprocess_data(X_valid)
X_test_processed = preprocess_data(X_test)


数据检查:
NaN 值数量: 0
无穷大值数量: 0
数据检查:
NaN 值数量: 0
无穷大值数量: 0
数据检查:
NaN 值数量: 0
无穷大值数量: 0


In [14]:
# 设置随机种子
np.random.seed(42)

# 定义模型
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42),
    'LightGBM': lgb.LGBMClassifier(random_state=42),
    'GBDT': GradientBoostingClassifier(random_state=42),
    'MLP': MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),  # 三层神经网络，节点数依次为256,128,64
        activation='relu',                   # ReLU激活函数
        solver='adam',                       # Adam优化器
        alpha=0.0001,                        # L2正则化参数
        batch_size='auto',
        learning_rate='adaptive',
        max_iter=1000,                       # 最大迭代次数
        early_stopping=True,                 # 启用早停
        validation_fraction=0.1,             # 用于早停的验证集比例
        random_state=42
    )
}


# 存储结果
results = {}

# 训练和评估每个模型
for name, model in models.items():
    print(f"\n训练 {name}...")
    
    # 训练模型
    model.fit(X_train_processed, y_train)
    
    # 在测试集上进行预测
    y_pred_proba = model.predict_proba(X_test_processed)[:, 1]

    # 保存预测结果
    results_df = pd.DataFrame({
        'op_id': df[df['dataset'] == 2][test_mask]['op_id'].values,
        'y_true': y_test,
        'y_pred_prob_0': y_pred_proba
    })
    
    # 为每个模型创建单独的文件
    results_path = f"results/{task_name}_test_preop_{name.lower().replace(' ', '_')}.csv"
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    results_df.to_csv(results_path, index=False)
    print(f"{name} 的预测结果已保存到: {results_path}")

    # 计算ROC和PR曲线
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    
    # 计算AUC值
    roc_auc = auc(fpr, tpr)
    pr_auc = auc(recall, precision)
    
    results[name] = {
        'fpr': fpr,
        'tpr': tpr,
        'precision': precision,
        'recall': recall,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc
    }
    
    print(f"{name} - ROC AUC: {roc_auc:.3f}, PR AUC: {pr_auc:.3f}")
    


训练 Random Forest...
Random Forest 的预测结果已保存到: results/postop_stroke_test_preop_random_forest.csv
Random Forest - ROC AUC: 0.917, PR AUC: 0.290

训练 XGBoost...
XGBoost 的预测结果已保存到: results/postop_stroke_test_preop_xgboost.csv
XGBoost - ROC AUC: 0.892, PR AUC: 0.311

训练 LightGBM...
[LightGBM] [Info] Number of positive: 700, number of negative: 70301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007276 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3169
[LightGBM] [Info] Number of data points in the train set: 71001, number of used features: 170
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009859 -> initscore=-4.609461
[LightGBM] [Info] Start training from score -4.609461
LightGBM 的预测结果已保存到: results/postop_stroke_test_preop_lightgbm.csv
LightGBM - ROC AUC: 0.917, PR AUC: 0.238

训练 GBDT...
GBDT 的预测结果已保存到: results/postop_stroke_test_pre